# StudyMate RAG pipeline

This notebook is intentionally dependency-light so it can run top-to-bottom on a fresh Python 3.10+ environment. It loads the curated Markdown corpus, creates section-aware chunks, builds a persisted TF-IDF vector index, tests retrieval, and records an evaluation table with ten questions. The runtime API can later swap the local embedding function for a sentence-transformers or Ollama-backed implementation without changing the citation contract.

In [ ]:
from pathlib import Path
from collections import Counter
from datetime import datetime, timezone
import json, math, re

ROOT = Path.cwd()
if not (ROOT / 'data').exists():
    ROOT = ROOT.parent
DOCS_DIR = ROOT / 'data' / 'documents'
INDEX_DIR = ROOT / 'data' / 'vector_store'
INDEX_DIR.mkdir(parents=True, exist_ok=True)
STOP_WORDS = {'a', 'an', 'and', 'are', 'be', 'by', 'for', 'from', 'how', 'in', 'is', 'it', 'of', 'on', 'or', 'the', 'to', 'what', 'when', 'why', 'with'}

def tokenize(text):
    return [t for t in re.findall(r'[a-z0-9]+', text.lower()) if len(t) > 2 and t not in STOP_WORDS]


## 2.1 Load and inspect

The corpus uses Markdown headings as stable section boundaries. This gives every chunk a human-readable citation instead of losing provenance during preprocessing.

In [ ]:
documents = []
for path in sorted(DOCS_DIR.glob('*.md')):
    text = path.read_text(encoding='utf-8').strip()
    documents.append({'id': path.stem, 'title': text.splitlines()[0].removeprefix('# ').strip(), 'text': text})
print(f'Loaded {len(documents)} documents from {DOCS_DIR}')
print('Formats:', sorted({path.suffix for path in DOCS_DIR.glob('*')}))


## 2.2 Chunking strategy

Each level-two heading creates one chunk. The source collection is short, so section-aware chunks are more useful than arbitrary fixed windows: the section title supplies a retrieval signal and the full paragraph remains readable as a citation.

In [ ]:
chunks = []
for document in documents:
    current = None
    for line in document['text'].splitlines()[1:]:
        if line.startswith('## '):
            if current and current['text'].strip():
                chunks.append(current)
            current = {'id': f"{document['id']}-{len(chunks)}", 'document_id': document['id'], 'title': document['title'], 'section': line[3:].strip(), 'text': ''}
        elif current is not None:
            current['text'] += line + ' '
    if current and current['text'].strip():
        chunks.append(current)
print(f'Created {len(chunks)} section-aware chunks')


## 2.3 Embeddings and vector store

For a reproducible local baseline, the embedding is a sparse TF-IDF vector. The same vectorizer is used for chunks and queries, and the vocabulary plus vectors are persisted to disk. In a production variant, replace this cell with sentence-transformers or a hosted embedding model while preserving the metadata fields.

In [ ]:
term_counts = [Counter(tokenize(chunk['title'] + ' ' + chunk['section'] + ' ' + chunk['text'])) for chunk in chunks]
document_frequency = Counter()
for counts in term_counts:
    document_frequency.update(counts.keys())
vocabulary = sorted(document_frequency)
N = len(chunks)

def embed(tokens):
    counts = Counter(tokens)
    vector = {}
    for term, count in counts.items():
        if term in document_frequency:
            idf = math.log((N + 1) / (document_frequency[term] + 1)) + 1
            vector[term] = (1 + math.log(count)) * idf
    norm = math.sqrt(sum(value * value for value in vector.values())) or 1
    return {term: value / norm for term, value in vector.items()}

def cosine(left, right):
    return sum(value * right.get(term, 0) for term, value in left.items())

for chunk, counts in zip(chunks, term_counts):
    chunk['embedding'] = embed(list(counts.elements()))
index = {'embedding_model': 'tfidf-local-v1', 'created_at': datetime.now(timezone.utc).isoformat(), 'chunks': chunks}
(INDEX_DIR / 'index.json').write_text(json.dumps(index, ensure_ascii=False, indent=2), encoding='utf-8')
(INDEX_DIR / 'manifest.json').write_text(json.dumps({'index_type': 'local-tfidf', 'embedding_model': 'tfidf-local-v1', 'chunk_count': len(chunks), 'document_count': len(documents), 'generated_by': 'notebooks/rag_pipeline.ipynb'}, indent=2), encoding='utf-8')
print(f'Persisted {len(chunks)} vectors to {INDEX_DIR / "index.json"}')


## 2.4 Retrieval and prompting

The retrieval function returns top-k chunks with scores and provenance. A production generation prompt should say: answer only from the supplied context, cite the section ids, and return an insufficient-evidence response when the best score is below the threshold.

In [ ]:
def retrieve(question, top_k=3):
    query_vector = embed(tokenize(question))
    ranked = sorted(((cosine(query_vector, chunk['embedding']), chunk) for chunk in chunks), key=lambda pair: pair[0], reverse=True)
    return ranked[:top_k]

sample = retrieve('Why should expensive resources load during application lifespan?')
[(round(score, 3), item['title'], item['section']) for score, item in sample]


## 2.6 Evaluation

The table below contains ten test questions across direct facts, comparisons, workflows, and one out-of-scope question. A hit means the expected document appears in the top three retrieved chunks. The out-of-scope question is expected to miss, which tests the refusal path.

In [ ]:
evaluation_questions = [
    ('What makes a Python function easy to test?', 'python-foundations'),
    ('When should I use a set instead of a list?', 'python-foundations'),
    ('How does FastAPI validate request bodies?', 'fastapi-practical'),
    ('Why use application lifespan?', 'fastapi-practical'),
    ('What is the retrieval-augmented loop?', 'rag-blueprint'),
    ('How should chunks preserve citations?', 'rag-blueprint'),
    ('What does cosine similarity measure?', 'embeddings-retrieval'),
    ('Which metrics evaluate a grounded assistant?', 'evaluation-playbook'),
    ('What belongs in a project README?', 'git-project-delivery'),
    ('What is the best recipe for chocolate cake?', None),
]
results = []
for question, expected_document in evaluation_questions:
    retrieved = retrieve(question)
    ids = [item['document_id'] for score, item in retrieved if score >= 0.08]
    hit = expected_document in ids if expected_document else not ids
    results.append({'question': question, 'retrieved_source': ids[0] if ids else 'insufficient evidence', 'correct': hit})
accuracy = sum(row['correct'] for row in results) / len(results)
print(json.dumps({'evaluation_score': round(accuracy, 2), 'results': results}, indent=2))
(INDEX_DIR / 'evaluation.json').write_text(json.dumps({'score': accuracy, 'results': results}, indent=2), encoding='utf-8')
